<a href="https://colab.research.google.com/github/normala127/NLP_Yelp_Review_Project/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Big Data Analytics Project
## Predicting Restaurant Failure: An NLP Driven Risk Assesment from Yelp Reviews

Students: Hatidza Imamovic, Asja Basovic

### 1. Dataset creation and environment setup

Three datasets are needed to create the final dataset which will be used for analysis and model training. These are:
- business.json: holds data about each restaurant
- review.json: holds all the reviews for each restaurant
- checkin.json: holds the dates of all checked in visits in a given restaurant

Each is loaded and then combined in regards to the business_id to ensure a correct join. The final output is saved as final.json.


In [1]:
!pip install pyspark
!apt-get install openjdk-17-jdk-headless -qq


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("project").getOrCreate()

print(spark)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
def show_shape(df):
  print((df.count(), len(df.columns)))

Loading the first dataset: business.json

In [6]:
df_business = spark.read.option("mode", "PERMISSIVE").json(r"/content/drive/MyDrive/yelp_academic_dataset_business.json")
df_business.printSchema()
df_business.show()

root
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: string (nullable = true)
 |    |-- Caters: string (nullable = true)
 |    |-- CoatCheck: string (nullable = true)
 |    |-- Corkage: string (nullable = true)
 |    |-- DietaryRestrictions: string (nullable = true)
 |    |-- DogsAllowed: string (nullable = true)
 |    |-- DriveThru: string (nullable = true)
 |    |-- GoodForDancing: str

Loading the second dataset: review.json

In [7]:
df_review = spark.read.json(r"/content/drive/MyDrive/yelp_academic_dataset_review.json")

In [8]:
show_shape(df_business)

(150346, 14)


In [9]:
show_shape(df_review)

(6990280, 9)


Joining df_review and df_business into joined_df

In [10]:
df_review.createOrReplaceTempView("review")
df_business.createOrReplaceTempView("business")

In [11]:
joined_df = spark.sql("""
SELECT t1.*, t2.*
FROM review t1
LEFT JOIN business t2 ON t2.business_id = t1.business_id
""")
joined_df.show(5)

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+-------+-------------+-----------+--------------------+-----------+------------+-----+-----+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|             address|          attributes|         business_id|          categories|        city|               hours|is_open|     latitude|  longitude|                name|postal_code|review_count|stars|state|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+-------+-------------+-----------+--------------------+-----------+------

In [12]:
joined_df = joined_df.drop(df_business['business_id'])

In [13]:
joined_df.columns # za obrisati

['business_id',
 'cool',
 'date',
 'funny',
 'review_id',
 'stars',
 'text',
 'useful',
 'user_id',
 'address',
 'attributes',
 'categories',
 'city',
 'hours',
 'is_open',
 'latitude',
 'longitude',
 'name',
 'postal_code',
 'review_count',
 'stars',
 'state']

In [14]:
show_shape(joined_df)

(6990280, 22)


Loading the third dataset: checkin.json

In [15]:
df_checkin = spark.read.option("mode", "PERMISSIVE").json(r"/content/drive/MyDrive/yelp_academic_dataset_checkin.json")
df_checkin.printSchema()
df_checkin.show()

root
 |-- business_id: string (nullable = true)
 |-- date: string (nullable = true)

+--------------------+--------------------+
|         business_id|                date|
+--------------------+--------------------+
|---kPU91CF4Lq2-Wl...|2020-03-13 21:10:...|
|--0iUa4sNDFiZFrAd...|2010-09-13 21:43:...|
|--30_8IhuyMHbSOcN...|2013-06-14 23:29:...|
|--7PUidqRWpRSpXeb...|2011-02-15 17:12:...|
|--7jw19RH9JKXgFoh...|2014-04-21 20:42:...|
|--8IbOsAAxjKRoYsB...|2015-06-06 01:03:...|
|--9osgUCSDUWUkoTL...|2015-06-13 02:00:...|
|--ARBQr1WMsTWiwOK...|2014-12-12 00:44:...|
|--FWWsIwxRwuw9vIM...|2010-09-11 16:28:...|
|--FcbSxK1AoEtEAxO...|2017-08-18 19:43:...|
|--LC8cIrALInl2vyo...|2017-01-12 19:10:...|
|--MbOh2O1pATkXa7x...|2013-04-21 01:52:...|
|--N9yp3ZWqQIm7DqK...|2012-10-06 20:46:...|
|--O3ip9NpXTKD4oBS...|2010-04-17 21:07:...|
|--OS_I7dnABrXvRCC...| 2018-05-11 18:23:36|
|--S43ruInmIsGrnnk...|2010-08-29 01:17:...|
|--SJXpAa0E-GCp2sm...|2014-04-06 22:23:...|
|--Sd93OFWITqDHifM...|2013-01-09 17

In [16]:
df_checkin_new=df_checkin.withColumnRenamed('date', 'date_checkin')
df_checkin_new.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- date_checkin: string (nullable = true)



Joining joined_df (review+business) and df_checkin into joined_df2

In [17]:
df_checkin_new.createOrReplaceTempView('checkin')
joined_df.createOrReplaceTempView('joined_df')

In [18]:
joined_df2 = spark.sql("""
SELECT t1.*, t2.date_checkin
FROM joined_df t1
JOIN checkin t2 ON t2.business_id = t1.business_id
""")
joined_df2.show(5)

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+-----+-----+--------------------+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|         address|          attributes|          categories|           city|               hours|is_open|  latitude|  longitude|             name|postal_code|review_count|stars|state|        date_checkin|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+-----+-----+--------------------+
|

In [19]:
joined_df2.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- cool: long (nullable = true)
 |-- date: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: 

Checking the distribution of opened and closed retaurants

In [20]:
df_isOpen=joined_df2.filter(joined_df2['is_open']==1)
df_isOpen.show()

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+-----+-----+--------------------+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|         address|          attributes|          categories|           city|               hours|is_open|  latitude|  longitude|             name|postal_code|review_count|stars|state|        date_checkin|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+----------------+--------------------+--------------------+---------------+--------------------+-------+----------+-----------+-----------------+-----------+------------+-----+-----+--------------------+
|

In [21]:
df_isClosed=joined_df2.filter(joined_df2['is_open']==0)
df_isClosed.show()

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------+--------------------+--------------------+------------+--------------------+-------+----------+-----------+--------------------+-----------+------------+-----+-----+--------------------+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|       address|          attributes|          categories|        city|               hours|is_open|  latitude|  longitude|                name|postal_code|review_count|stars|state|        date_checkin|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------+--------------------+--------------------+------------+--------------------+-------+----------+-----------+--------------------+-----------+------------+-----+-----+--------------------+
|-0eUa8

In [22]:
show_shape(df_isOpen) # 80% is open


(5616746, 23)


In [23]:
show_shape(df_isClosed) # 20% is closed

(1180155, 23)


Dropping a part of open restaurants based on business_id to balance out the classes

In [24]:
from pyspark.sql.functions import col, hash

# id-level labels
id_labels = joined_df2.select("business_id", "is_open").distinct()

majority_ids = id_labels.filter(col("is_open") == 1) \
    .withColumn("keep", (hash("business_id") % 10) < 2)  # keep 20%

minority_ids = id_labels.filter(col("is_open") == 0) \
    .withColumn("keep", col("is_open").isNotNull())  # keep all

ids_to_keep = majority_ids.filter("keep").union(
    minority_ids.select("business_id", 'is_open', 'keep')
)

balanced_df = joined_df2.join(ids_to_keep.select("business_id"),
                      "business_id")

In [25]:
show_shape(balanced_df)

(4552751, 23)


In [26]:
balanced_df.show()

+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------+--------------------+--------------------+------------+--------------------+-------+----------+-----------+--------------------+-----------+------------+-----+-----+--------------------+
|         business_id|cool|               date|funny|           review_id|stars|                text|useful|             user_id|       address|          attributes|          categories|        city|               hours|is_open|  latitude|  longitude|                name|postal_code|review_count|stars|state|        date_checkin|
+--------------------+----+-------------------+-----+--------------------+-----+--------------------+------+--------------------+--------------+--------------------+--------------------+------------+--------------------+-------+----------+-----------+--------------------+-----------+------------+-----+-----+--------------------+
|-0eUa8

In [27]:
majority_ids = id_labels.filter(col("is_open") == 1) \
    .withColumn("keep", (hash("business_id") % 10) < 2)  # keep 20%

minority_ids = id_labels.filter(col("is_open") == 0) \
    .withColumn("keep", col("is_open").isNotNull())  # keep all

In [30]:
show_shape(majority_ids)

(103160, 3)


In [31]:
show_shape(minority_ids)

(28770, 3)


Renaming columns and dropping uneeded columns

In [32]:
balanced_df.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- cool: long (nullable = true)
 |-- date: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: 

In [33]:
cols=balanced_df.columns
stars_columns=[i for i, c in enumerate(cols) if c=='stars']

cols[stars_columns[0]]='stars_review'
cols[stars_columns[1]]='stars_business'

balanced_df=balanced_df.toDF(*cols)

In [34]:
balanced_df.columns

['business_id',
 'cool',
 'date',
 'funny',
 'review_id',
 'stars_review',
 'text',
 'useful',
 'user_id',
 'address',
 'attributes',
 'categories',
 'city',
 'hours',
 'is_open',
 'latitude',
 'longitude',
 'name',
 'postal_code',
 'review_count',
 'stars_business',
 'state',
 'date_checkin']

In [35]:
final_df=balanced_df.drop(*['cool', 'funny', 'useful', 'latitude', 'longitude', 'postal_code', 'attributes', 'hours'])

In [36]:
final_df.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars_review: double (nullable = true)
 |-- text: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- city: string (nullable = true)
 |-- is_open: long (nullable = true)
 |-- name: string (nullable = true)
 |-- review_count: long (nullable = true)
 |-- stars_business: double (nullable = true)
 |-- state: string (nullable = true)
 |-- date_checkin: string (nullable = true)



In [37]:
show_shape(final_df)

(4552751, 15)


Saving the final dataset to the drive as a json file

In [38]:
#final_df.write.mode("overwrite").parquet("/content/drive/MyDrive/dataprj/dataset2/dataset.parquet")

In [39]:
#final_df.write.mode("overwrite").parquet("/content/test_parquet")

### 2. Preprocessing

Firstly, on a global level, null and duplicate values were dropped.

This section covers:
- lowercase,
- keep only letters (from all languages) and spaces,
- remove extra spaces,
- remove private information,
- emojis, urls.

It creates a pipeline for TF-IDF and for semantic analysis as slightly different cleaning techniques are used for each.

The section also uses n-grams, and handles "not" negation effectively.


In [75]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF, NGram, VectorAssembler

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [42]:
df = final_df.select("*")

In [47]:
from pyspark.sql.functions import when, col, count, sum

null_counts = df.select([count(when(col(c).isNull(), c).alias(c)) for c in df.columns])

null_counts.show()

+--------------------------------------------------------------------------+-----------------------------------------------------+--------------------------------------------------------------------+-----------------------------------------------------------------------------+-----------------------------------------------------+--------------------------------------------------------------+--------------------------------------------------------------+-----------------------------------------------------------------------+-----------------------------------------------------+--------------------------------------------------------------+-----------------------------------------------------+-----------------------------------------------------------------------------+-----------------------------------------------------------------------------------+--------------------------------------------------------+-----------------------------------------------------------------------------+
|c

In [48]:
df_nulls = df.filter(df['categories'].isNull())
df_nulls.limit(10)

+--------------------+-------------------+--------------------+------------+--------------------+--------------------+-------------------+----------+------------+-------+--------------------+------------+--------------+-----+--------------------+
|         business_id|               date|           review_id|stars_review|                text|             user_id|            address|categories|        city|is_open|                name|review_count|stars_business|state|        date_checkin|
+--------------------+-------------------+--------------------+------------+--------------------+--------------------+-------------------+----------+------------+-------+--------------------+------------+--------------+-----+--------------------+
|eer6JTQeGaLbOwuJu...|2009-06-03 21:23:58|dbVPM79PiB3LbEd2C...|         5.0|50's style diner ...|dbrGVOcscXUQjQGi_...|      201 S Main St|      NULL|Ashland City|      0|          Stratton's|           5|           4.5|   TN| 2010-08-14 18:21:07|
|eer6JTQeGaL

In [ ]:
# TODO visaulaization

In [49]:
df = df.fillna({'categories': 'Unknown'})

In [50]:
df = df.dropDuplicates(['text'])
show_shape(df)

(4542946, 15)


In [ ]:
def clean_sentiment(df):
  return df.withColumn("text_cleaned",
          F.trim(
            F.regexp_replace(
              F.lower(F.col("text")),
              r"http\S+|www\S+|\S+@\S+", " "
            ),
          )
      ).withColumn("text_cleaned", F.regexp_replace(F.col("text_cleaned"), r"\s+", " "))


In [69]:
def clean_tfidf_fast(df):

    return df.withColumn("text_cleaned",
        F.trim(
            F.regexp_replace(
                F.regexp_replace(
                    F.lower(F.col("text")),
                    r"http\S+|www\S+|\S+@\S+", " "
                ),
                r'[^\p{L}\s]+', " "
            )
        )
    ).withColumn("text_cleaned", F.regexp_replace(F.col("text_cleaned"), r"\s+", " "))

In [70]:
df=clean_tfidf_fast(df)
#df.show()

+--------------------+-------------------+--------------------+------------+--------------------+--------------------+--------------------+--------------------+----------------+-------+--------------------+------------+--------------+-----+--------------------+--------------------+
|         business_id|               date|           review_id|stars_review|                text|             user_id|             address|          categories|            city|is_open|                name|review_count|stars_business|state|        date_checkin|        text_cleaned|
+--------------------+-------------------+--------------------+------------+--------------------+--------------------+--------------------+--------------------+----------------+-------+--------------------+------------+--------------+-----+--------------------+--------------------+
|7X2U_02n8IOGKILtg...|2017-03-16 13:38:10|MKA-mplFhw6BPuZ9F...|         5.0|"Can I keep this ...|7Tz2XdgB9w887eSPe...|    2038 McKelvey Rd|Local Servic

In [ ]:
#todo partitioning

In [ ]:
updated_stops = [w for w in default_stops if w != 'not']

def tfidf(vocab_size =10000, min_df = 500):

  tokenizer = RegexTokenizer(
      inputCol="text",
      outputCol="tokens_raw",
      pattern=r"\W+",
      gaps=True,
      toLowercase=True,
      minTokenLength=1
  )

  remover = StopWordsRemover(inputCol="tokens_raw", outputCol="filtered_tokens", stopWords=updated_stops)

  ngram = NGram(n=2, inputCol="filtered_tokens", outputCol="bigrams")

  uni_vectorizer = CountVectorizer(
        inputCol="filtered_tokens",
        outputCol="uni_count_features",
        vocabSize=vocab_size,
        minDF=min_df
    )

  bi_vectorizer = CountVectorizer(
      inputCol = 'bigrams',
      outputCol = "bi_count_features",
      vocabSize = vocab_size,
      minDF=min_df
  )

  assembler = VectorAssembler(
        inputCols=["uni_count_features", "bi_count_features"],
        outputCol="combined_counts"
    )

  idf = IDF(inputCol="combined_counts", outputCol="features", minDocFreq=min_df)

  return  [tokenizer, remover, ngram, uni_vectorizer, bi_vectorizer, assembler, idf]


In [ ]:
remover = StopWordsRemover()
default_stops = remover.getStopWords()
updated_stops = [w for w in default_stops if w != 'not']

def tfidf(vocab_size =10000, min_df = 500):

  tokenizer = RegexTokenizer(
      inputCol="text",
      outputCol="tokens_raw",
      pattern=r"\W+",
      gaps=True,
      toLowercase=True,
      minTokenLength=1
  )

  remover = StopWordsRemover(inputCol="tokens_raw", outputCol="filtered_tokens", stopWords=updated_stops)

  ngram = NGram(n=2, inputCol="filtered_tokens", outputCol="bigrams")

  uni_vectorizer = CountVectorizer(
        inputCol="filtered_tokens",
        outputCol="uni_count_features",
        vocabSize=vocab_size,
        minDF=min_df
    )

  bi_vectorizer = CountVectorizer(
      inputCol = 'bigrams',
      outputCol = "bi_count_features",
      vocabSize = vocab_size,
      minDF=min_df
  )

  assembler = VectorAssembler(
        inputCols=["uni_count_features", "bi_count_features"],
        outputCol="combined_counts"
    )

  idf = IDF(inputCol="combined_counts", outputCol="features", minDocFreq=min_df)

  return  [tokenizer, remover, ngram, uni_vectorizer, bi_vectorizer, assembler, idf]


In [ ]:
remover = StopWordsRemover()
default_stops = remover.getStopWords()
updated_stops = [w for w in default_stops if w != 'not']

def tfidf(vocab_size =10000, min_df = 500):

  tokenizer = RegexTokenizer(
      inputCol="text",
      outputCol="tokens_raw",
      pattern=r"\W+",
      gaps=True,
      toLowercase=True,
      minTokenLength=1
  )

  remover = StopWordsRemover(inputCol="tokens_raw", outputCol="filtered_tokens", stopWords=updated_stops)

  ngram = NGram(n=2, inputCol="filtered_tokens", outputCol="bigrams")

  uni_vectorizer = CountVectorizer(
        inputCol="filtered_tokens",
        outputCol="uni_count_features",
        vocabSize=vocab_size,
        minDF=min_df
    )

  bi_vectorizer = CountVectorizer(
      inputCol = 'bigrams',
      outputCol = "bi_count_features",
      vocabSize = vocab_size,
      minDF=min_df
  )

  assembler = VectorAssembler(
        inputCols=["uni_count_features", "bi_count_features"],
        outputCol="combined_counts"
    )

  idf = IDF(inputCol="combined_counts", outputCol="features", minDocFreq=min_df)

  return  [tokenizer, remover, ngram, uni_vectorizer, bi_vectorizer, assembler, idf]


In [51]:
from pyspark.sql import functions as F

unique_bus_df = df.select("business_id", "is_open").distinct()

fractions = {0: 0.8, 1: 0.8} # 80% of closed (0) and 80% of open (1)

train_ids = unique_bus_df.sampleBy("is_open", fractions, seed=42)

test_ids = unique_bus_df.join(train_ids, on="business_id", how="left_anti")

train_df = df.join(train_ids.select("business_id"), on="business_id", how="inner")
test_df = df.join(test_ids.select("business_id"), on="business_id", how="inner")

In [45]:
#show_shape(train_df)

(3660048, 15)


In [46]:
#show_shape(test_df)

(909058, 15)


### 3. Feature engineering

In [74]:
remover = StopWordsRemover()
default_stops = remover.getStopWords()
updated_stops = [w for w in default_stops if w != 'not']

def tfidf(vocab_size =10000, min_df = 500):

  tokenizer = RegexTokenizer(
      inputCol="text",
      outputCol="tokens_raw",
      pattern=r"\W+",
      gaps=True,
      toLowercase=True,
      minTokenLength=1
  )

  remover = StopWordsRemover(inputCol="tokens_raw", outputCol="filtered_tokens", stopWords=updated_stops)

  ngram = NGram(n=2, inputCol="filtered_tokens", outputCol="bigrams")

  uni_vectorizer = CountVectorizer(
        inputCol="filtered_tokens",
        outputCol="uni_count_features",
        vocabSize=vocab_size,
        minDF=min_df
    )

  bi_vectorizer = CountVectorizer(
      inputCol = 'bigrams',
      outputCol = "bi_count_features",
      vocabSize = vocab_size,
      minDF=min_df
  )

  assembler = VectorAssembler(
        inputCols=["uni_count_features", "bi_count_features"],
        outputCol="combined_counts"
    )

  idf = IDF(inputCol="combined_counts", outputCol="features", minDocFreq=min_df)

  return  [tokenizer, remover, ngram, uni_vectorizer, bi_vectorizer, assembler, idf]


In [76]:
tfidf_pipeline = tfidf()

sentinemnt analysis
Clean other columns
EDA
FE
Build and hypertune models (check literature)
Metrics and eval